# Model Selection Demo

This notebook inspects available language-pair models from configuration and builds a practical model selection matrix for different use cases.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'config.py').exists() and (candidate / 'models').exists():
            return candidate
    raise RuntimeError('Could not locate Machine_Translation project root.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from config import TranslationConfig

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\Nikolai\OneDrive\Desktop\Portfolio\CS_Language_Portfolio\projects\Machine_Translation


In [2]:
config = TranslationConfig.default()
rows = []
for (src, tgt), pair_cfg in config.language_pairs.items():
    rows.append({
        'pair': f'{src}->{tgt}',
        'source_lang': src,
        'target_lang': tgt,
        'model_name': pair_cfg.model_name,
        'max_length': pair_cfg.max_length,
        'beam_size_default': config.num_beams,
        'device_default': config.device
    })

models_df = pd.DataFrame(rows).sort_values('pair').reset_index(drop=True)
models_df

,pair,source_lang,target_lang,model_name,max_length,beam_size_default,device_default
0,de->en,de,en,Helsinki-NLP/Opus-MT-de-en,512,5,cuda
1,en->de,en,de,Helsinki-NLP/Opus-MT-en-de,512,5,cuda
2,en->ru,en,ru,Helsinki-NLP/Opus-MT-en-ru,512,5,cuda
3,ru->en,ru,en,Helsinki-NLP/Opus-MT-ru-en,512,5,cuda


In [3]:
def scenario_score(model_name: str, scenario: str) -> float:
    # Lightweight heuristic for portfolio planning, not measured latency.
    base = 0.7
    if 'Helsinki-NLP' in model_name:
        base += 0.1

    scenario_adjustments = {
        'quality_first': 0.20,
        'balanced': 0.10,
        'latency_first': -0.05
    }
    return round(base + scenario_adjustments.get(scenario, 0.0), 2)

scenarios = ['quality_first', 'balanced', 'latency_first']
matrix_rows = []
for _, row in models_df.iterrows():
    for scenario in scenarios:
        matrix_rows.append({
            'pair': row['pair'],
            'model_name': row['model_name'],
            'scenario': scenario,
            'selection_score': scenario_score(row['model_name'], scenario)
        })

selection_df = pd.DataFrame(matrix_rows).sort_values(['scenario', 'selection_score'], ascending=[True, False])
selection_df

,pair,model_name,scenario,selection_score
1,de->en,Helsinki-NLP/Opus-MT-de-en,balanced,0.90
4,en->de,Helsinki-NLP/Opus-MT-en-de,balanced,0.90
7,en->ru,Helsinki-NLP/Opus-MT-en-ru,balanced,0.90
10,ru->en,Helsinki-NLP/Opus-MT-ru-en,balanced,0.90
2,de->en,Helsinki-NLP/Opus-MT-de-en,latency_first,0.75
5,en->de,Helsinki-NLP/Opus-MT-en-de,latency_first,0.75
8,en->ru,Helsinki-NLP/Opus-MT-en-ru,latency_first,0.75
11,ru->en,Helsinki-NLP/Opus-MT-ru-en,latency_first,0.75
0,de->en,Helsinki-NLP/Opus-MT-de-en,quality_first,1.00
3,en->de,Helsinki-NLP/Opus-MT-en-de,quality_first,1.00


In [4]:
recommended = (
    selection_df.sort_values('selection_score', ascending=False)
    .groupby(['pair', 'scenario'], as_index=False)
    .first()
    .sort_values(['pair', 'scenario'])
)

recommended

,pair,scenario,model_name,selection_score
0,de->en,balanced,Helsinki-NLP/Opus-MT-de-en,0.90
1,de->en,latency_first,Helsinki-NLP/Opus-MT-de-en,0.75
2,de->en,quality_first,Helsinki-NLP/Opus-MT-de-en,1.00
3,en->de,balanced,Helsinki-NLP/Opus-MT-en-de,0.90
4,en->de,latency_first,Helsinki-NLP/Opus-MT-en-de,0.75
5,en->de,quality_first,Helsinki-NLP/Opus-MT-en-de,1.00
6,en->ru,balanced,Helsinki-NLP/Opus-MT-en-ru,0.90
7,en->ru,latency_first,Helsinki-NLP/Opus-MT-en-ru,0.75
8,en->ru,quality_first,Helsinki-NLP/Opus-MT-en-ru,1.00
9,ru->en,balanced,Helsinki-NLP/Opus-MT-ru-en,0.90


In [5]:
# Optional live translation smoke test. Set RUN_LIVE_TEST=True only when model dependencies are installed.
RUN_LIVE_TEST = False

if RUN_LIVE_TEST:
    from api import translate
    result = translate('Good morning, how are you?', 'en', 'de')
    print('Translation:', result.text)
    print('Metadata:', result.metadata)
else:
    print('Skipped live translation test. Toggle RUN_LIVE_TEST to True to execute.')

Skipped live translation test. Toggle RUN_LIVE_TEST to True to execute.


## Recommendation Pattern

- Use the configured Opus-MT defaults for baseline experiments.
- Keep scenario-based scorecards for product constraints (quality vs. latency).
- Confirm with evaluation notebook metrics before final model choice.